## Model Validation: Can an Open-Source Model Match GPT-4.1?

### 1. Goal

The original study (Jacobs et al., 2025) used GPT-4.1 to classify papers by ecosystem service and technological trajectory (Replace / Enhance / Support). To keep the platform low-cost and easy to maintain as the corpus grows, we tested whether an open-source model could match GPT-4.1 on the same classification task, using the 100-paper validation set from the original study.

Two open-source models were tested — **Qwen-2.5-72B-instruct** and **DeepSeek-R1-distill-llama-70B** — across three prompt versions, to see whether the gap (if any) was a model-capability issue or a prompt-design issue.

### 2. Approach

**1. Build a fair, reproducible scoring function.**
The original accuracy was scored manually by the authors, which inevitably involves some subjective judgment that we cannot fully reproduce. We built a rule-based scoring function (with service-name normalization to remove formatting-only mismatches) and verified it against GPT-4.1: it reproduces 82.0 vs the authors' reported 84.5 — close enough to serve as a consistent yardstick across all models.

**2. Baseline — original prompt.**
Both models were run with the unmodified original prompt.
Result: Qwen 72, DeepSeek 73, vs GPT-4.1 82 (~10-point gap for both).

**3. Diagnose the gap.**
Error analysis showed that both models were overly conservative: both models wrongly marked true-Y papers as N, concentrated in Biochemicals,
Disease Regulation, and Fibre/Hide/Wood. Both models read "ecosystem service" too literally and missed that bio-inspired drugs, medical materials, and biomimetic synthesis count functionally.

**4. Prompt V1 — positive few-shot examples.**
Added a clarification that an ES contribution is functional, not locational, plus three real missed cases as positive few-shot examples.
Result: over-conservatism was fixed (missed cases dropped sharply), but this overcorrected into a new problem — both models started marking many true-N papers as Y. Scores dropped to 55 for both models.

**5. Prompt V2 — balanced few-shot examples.**
Added negative examples alongside the positive ones, to teach the models where the boundary actually sits (drawn from the false positives in V1).
Result: the false positives were corrected, but the original over-conservative errors returned almost exactly. Scores settled back to ~70–72 for both models — essentially the same as the baseline.

### 3. Outcome

Across three prompt versions, Qwen-2.5-72B and DeepSeek-R1-70B produced nearly identical scores (72/55/70 vs 73/55/72), despite being very different models. This strongly suggests the ~10-point gap is **not a model-capability limitation**, but a **hard ceiling set by the data itself** — specifically, a genuinely ambiguous boundary (does a bio-inspired synthesis or material count as an ES contribution?) where the original human labels are themselves inconsistent (e.g. two near-identical bio-inspired synthesis papers were labeled Y and N respectively).

Prompt engineering could make the models more conservative or more permissive, but it could not push either model past this ceiling.

**DThe final choice of model for the production pipeline will be discussed with the supervisors.** — see the accompanying email for
the three candidate directions (open-source model, continued use of GPT, or a hybrid approach).

In [1]:
import os
import pandas as pd
from openai import OpenAI
from tqdm import tqdm  
import time

import json
import re

# more affordable one
TARGET_MODEL = "qwen/qwen-2.5-72b-instruct" 

client = OpenAI(
    api_key="sk-or", # OpenRouter API key — please don't leak it :) 
    base_url="https://openrouter.ai/api/v1"
)

# read the datasets
df_test = pd.read_excel("Project ME_validation 100.xlsx", sheet_name = 'Input_WoS_100')
df_validate = pd.read_excel("Project ME_validation 100.xlsx", sheet_name = 'Manual validation')

In [2]:
df_test.columns

Index(['No. (number used only for testing stage)',
       'No. (number in the final corpus)', 'Publication Year', 'Article Title',
       'Author Keywords', 'Keywords Plus', 'Abstract'],
      dtype='str')

In [3]:
df_validate.columns

Index(['No. (number used only for testing stage)',
       'No. (number in the final corpus)', 'Decision (human)',
       'Category (human)', 'EcosystemService (human)', 'ReviewFlag (human)',
       'Decision (gpt)', 'Category (gpt)', 'EcosystemService (gpt)',
       'ReviewFlag (gpt)', 'accuracy score ', 'note'],
      dtype='str')

In [4]:
df_validate['note']

0                                                    NaN
1      All review papers are excluded, so if GPT corr...
2                                                    NaN
3                                                    NaN
4              Decision (gpt) and Category (gpt) correct
                             ...                        
97                              or Inspiration/Education
98                                                   NaN
99                                                   NaN
100                                                  NaN
101                                                  NaN
Name: note, Length: 102, dtype: str

## 1. Use the original prompt to see how Qwen performs

In [5]:

SYSTEM_PROMPT = """ You are an Ecosystem Service expert and a dedicated assistant designed to classify research articles (titles, keywords, abstracts given) based on the following instructions:
1. **Ecosystem Service Technology Analysis:**
Determine if the abstract describes a technological intervention that contributes to one or more of the following ecosystem services:
**Provisioning—Products obtained from ecosystems (Existing commercial market):**
- Biodiversity—The number of different species
- Food—Ingredients derived from wild and domesticated habitats
- Potable Water—Fresh water that is safe to consume
- Fuel—Materials used to generate energy
- Fibre/Hide/Wood—Materials used for clothing or construction
- Biochemicals—Molecules used in medicine
**Cultural—Benefits to quality of life and community (Existing commercial market):**
- Spiritual—Supporting the spiritual lives of people
- Recreation—Supporting the physical and mental health of people
- Aesthetic—The mental and physical health benefits of natural beauty
- Inspiration/Education—Art, music, literature, architecture, and engineering design
- Cultural Heritage—Value placed upon landscapes
- Cultural Identity—Societal identity regulated by the ecosystem (e.g., nomadic herding)
**Regulating—Benefits obtained by regulating ecosystem processes (Most amenable to technological replacement):**
- Atmospheric Regulation—Production and consumption of essential molecules (e.g., oxygen)
- Climate Regulation—Stabilization of climatic conditions
- Coastline Regulation—Stabilization of coastal lands (e.g., mangroves and reefs)
- Disease Regulation—Natural systems that reduce human disease or disease vectors
- Water Regulation—Timing and volume of water distribution across the landscape
- Waste Treatment—Filtering and treatment of waste products (incl. organics and water)
- Pollination—Distribution of pollen for the purpose of plant reproduction
**Supporting—Services that are not necessary for all other ecosystem services (Least amenable to technological replacement):**
- Soil Formation—The creation of new soil
- Nutrient Cycling—The movement of nutrients through the ecosystems
- Primary Production—The creation of sugars from sunlight
For this part:
**Decision:** Output “Y” if the abstract explicitly describes a practical technological method that contributes to one or more of these services; otherwise, output “N”.
**Category:**
If Decision is “Y”, choose **one** of the following:
- **”Support”** assists or maintains an existing natural process without intensifying it. Example: “Adding baffles so river flow still scours sediment but a little more efficiently.”
- **”Enhance”** significantly boosts the efficiency or scale of a natural process while still relying on that process. Example: “Embedding enzymes in a filter to double the nitrification rate; process still needs microbes.”
- **”Replace”** creates an artificial substitute that operates independently of the natural process. Example: “A photocatalytic panel that fixes nitrogen from air in total isolation from biological pathways.”
(If uncertain between Enhance and Replace, choose Enhance.)
Leave blank if Decision is “N”
**EcosystemService:** If Decision is “Y”, provide the exact ecosystem service from the list.
**Technology:** If Decision is “Y”, provide a concise short name for the technology used.
2. **Review Paper Detection:**
Determine if the abstract indicates that the article is a review paper. If the abstract contains phrases like “review”, “survey”, “meta-analysis”, or other similar indicators, then:
- **ReviewFlag:** Set to “review”.
Otherwise, leave this field blank.

**Output Format (Strict JSON Only)**
Your output must be in JSON format only, following this structure:
“Decision”: “Y” or “N”,
“Category”: “Enhance” or “Replace” (leave blank if Decision = “N”),
“EcosystemService”: “(exact ecosystem service from the list)” (leave blank if Decision = “N”),
“Technology”: “(concise short name of the technology)” (leave blank if Decision = “N”),
“ReviewFlag”: “review” or “”
}"""

In [28]:

def call_with_retry(client, model, system_prompt, content, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": content}
                ],
                temperature=0.1,
                max_tokens=512,  
            )
            if (response is None or response.choices is None
                    or len(response.choices) == 0
                    or response.choices[0].message.content is None):
                raise ValueError("Empty or malformed response")
            return response.choices[0].message.content, "Success"
        except Exception as e:
            wait = 2 ** attempt
            if attempt < max_retries - 1:
                time.sleep(wait)
            else:
                return str(e), "Failed"

In [6]:

results = []

for index, row in tqdm(df_test.iterrows(), total=len(df_test),
                       desc=f"Benchmarking {TARGET_MODEL}"):
    paper_id      = str(row['No. (number used only for testing stage)'])
    title_text    = str(row['Article Title'])
    keywords_text = str(row['Author Keywords'])
    abstract_text = str(row['Abstract'])

    combined_content = (f"Title: {title_text}\n"
                        f"Keywords: {keywords_text}\n"
                        f"Abstract: {abstract_text}")

    raw_output, status = call_with_retry(
        client, TARGET_MODEL, SYSTEM_PROMPT, combined_content
    )
    results.append({"wos_id": paper_id, "qwen_output": raw_output, "status": status})

    time.sleep(0.6)   

df_predictions = pd.DataFrame(results)

# sanity check
print(df_predictions['status'].value_counts())

Benchmarking qwen/qwen-2.5-72b-instruct: 100%|█| 100/100 [07:37<00:00,  4.58s/it

status
Success    100
Name: count, dtype: int64


In [7]:

# 1. JSON parser — Extracting fields from Qwen's raw output
def parse_full_qwen_json(output_str):
    try:
        match = re.search(r'\{.*\}', str(output_str), re.DOTALL)
        data = json.loads(match.group()) if match else json.loads(output_str)
        return pd.Series({
            'qwen_decision': str(data.get("Decision", "")).strip(),
            'qwen_category': str(data.get("Category", "")).strip(),
            'qwen_service':  str(data.get("EcosystemService", "")).strip(),
            'qwen_review':   str(data.get("ReviewFlag", "")).strip().lower(),
        })
    except Exception:
        return pd.Series({'qwen_decision': None, 'qwen_category': None,
                          'qwen_service': None, 'qwen_review': None})

df_predictions[['qwen_decision', 'qwen_category', 'qwen_service', 'qwen_review']] = \
    df_predictions['qwen_output'].apply(parse_full_qwen_json)

# 2.Merging prediction results with human-verified data

col_id_val = 'No. (number used only for testing stage)'

# clean IDs on both sides to ensure merge alignment (remove .0 suffix and whitespaces)
df_predictions['wos_id'] = (
    df_predictions['wos_id'].astype(str)
    .str.replace(r'\.0$', '', regex=True).str.strip()
)
df_validate[col_id_val] = (
    df_validate[col_id_val].astype(str)
    .str.replace(r'\.0$', '', regex=True).str.strip()
)

# Exclude summary rows like row 101 (where human columns are all empty)
df_validate_clean = df_validate[df_validate['Decision (human)'].notna()].copy()

df_merged = pd.merge(
    df_validate_clean,
    df_predictions,
    left_on=col_id_val,
    right_on='wos_id',
    how='inner'
)

In [27]:
# df_merged.to_excel('predictions_merged.xlsx', index=False)

In [8]:

# 3. Define normalization function to remove false errors caused by formatting
def normalize_service(s):
    """Normalize service names by removing whitespace, casing, and delimiter differences."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r'\s*/\s*', '/', s)   # "Fibre / Hide" -> "fibre/hide"
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

def normalize_simple(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    return str(s).strip().lower()


# 4. Unified scoring function: pass 4 prediction column names and return the row score
def score_row(row, dec_col, cat_col, srv_col, rev_col):
    h_dec = normalize_simple(row['Decision (human)'])
    h_cat = normalize_simple(row['Category (human)'])
    h_srv = normalize_service(row['EcosystemService (human)'])
    h_rev = normalize_simple(row['ReviewFlag (human)'])

    p_dec = normalize_simple(row[dec_col])
    p_cat = normalize_simple(row[cat_col])
    p_srv = normalize_service(row[srv_col])
    p_rev = normalize_simple(row[rev_col])

    # Rule 1: Exempt review papers from evaluation
    if 'review' in h_rev and 'review' in p_rev:
        return 1.0
    # Rule 2: Both Sides Concur on 'N'
    if h_dec == 'n' and p_dec == 'n':
        return 1.0
    # Rule 3: Both sides predict Y, layered scoring
    if h_dec == 'y' and p_dec == 'y':
        if h_srv == p_srv and h_cat == p_cat:
            return 1.0
        elif h_srv == p_srv and h_cat != p_cat:
            return 0.5
        else:
            return 0.0
    # Rule 4: Decision mismatch
    return 0.0

In [9]:

# GPT-4.1 scoring — Verifying alignment between the scoring function and human labels
df_merged['gpt_score'] = df_merged.apply(
    lambda r: score_row(r, 'Decision (gpt)', 'Category (gpt)',
                        'EcosystemService (gpt)', 'ReviewFlag (gpt)'),
    axis=1
)
gpt_final = df_merged['gpt_score'].mean() * 100

# Baseline: Author's Human Score Mean (Excluding Summary Rows)
author_final = df_validate_clean['accuracy score '].dropna().mean() * 100

# Qwen scoring
df_merged['qwen_score'] = df_merged.apply(
    lambda r: score_row(r, 'qwen_decision', 'qwen_category',
                        'qwen_service', 'qwen_review'),
    axis=1
)
qwen_final = df_merged['qwen_score'].mean() * 100

In [14]:

# summary
print("="*60)
print("Model Benchmarking (rule-based scoring)")
print("="*60)
print(f"Author's manual score for GPT-4.1 : {author_final:5.2f}")
print(f"My rule-based score for GPT-4.1   : {gpt_final:5.2f}")
print(f"My rule-based score for Qwen      : {qwen_final:5.2f}")
print(f"GPT-4.1 vs Qwen gap               : {gpt_final - qwen_final:5.2f}")
print("="*60)

print("\nQwen score distribution:")
print(df_merged['qwen_score'].value_counts().sort_index())

Model Benchmarking (rule-based scoring)
Author's manual score for GPT-4.1 : 84.50
My rule-based score for GPT-4.1   : 82.00
My rule-based score for Qwen      : 72.00
GPT-4.1 vs Qwen gap               : 10.00

Qwen score distribution:
qwen_score
0.0    25
0.5     6
1.0    69
Name: count, dtype: int64


In [15]:
# Error diagnosis 
def diagnose(row):
    if row['qwen_score'] == 1.0:
        return 'correct'
    h_dec = normalize_simple(row['Decision (human)'])
    q_dec = normalize_simple(row['qwen_decision'])
    h_srv = normalize_service(row['EcosystemService (human)'])
    q_srv = normalize_service(row['qwen_service'])
    if h_dec == 'y' and q_dec == 'n':
        return 'miss_Y_as_N (over-conservative)'
    if h_dec == 'n' and q_dec == 'y':
        return 'false_Y'
    if h_dec == 'y' and q_dec == 'y':
        return 'wrong_service' if h_srv != q_srv else 'wrong_category'
    return 'other'

df_merged['error_type'] = df_merged.apply(diagnose, axis=1)
print("\n Qwen error breakdown:")
print(df_merged['error_type'].value_counts())


 Qwen error breakdown:
error_type
correct                            69
miss_Y_as_N (over-conservative)    19
wrong_category                      6
wrong_service                       4
false_Y                             2
Name: count, dtype: int64


In [19]:
# Check the actual values of qwen_decision in df_predictions
print("=== qwen_decision value distribution (including null values)===")
print(df_predictions['qwen_decision'].value_counts(dropna=False))

# Inspect df_merged
print("\n=== Value Distribution of qwen_decision ===")
print(df_merged['qwen_decision'].value_counts(dropna=False))

# Inspect a few raw qwen_output samples to verify if the parser successfully extracted the fields
print("\n=== First 3 raw outputs ===")
for x in df_predictions['qwen_output'].head(3):
    print(repr(x)[:300])
    print("---")

# Check the total number of matched rows post-merge
print(f"\ndf_predictions rows: {len(df_predictions)}")
print(f"df_validate_clean rows: {len(df_validate_clean)}")
print(f"df_merged rows: {len(df_merged)}")

=== qwen_decision value distribution (including null values)===
qwen_decision
N    77
Y    23
Name: count, dtype: int64

=== Value Distribution of qwen_decision ===
qwen_decision
N    77
Y    23
Name: count, dtype: int64

=== First 3 raw outputs ===
'{\n  "Decision": "N",\n  "Category": "",\n  "EcosystemService": "",\n  "Technology": "",\n  "ReviewFlag": ""\n}'
---
'{\n  "Decision": "Y",\n  "Category": "Enhance",\n  "EcosystemService": "Potable Water",\n  "Technology": "Bioinspired and Biomimetic Membranes (BBMs)",\n  "ReviewFlag": "review"\n}'
---
'{\n  "Decision": "N",\n  "Category": "",\n  "EcosystemService": "",\n  "Technology": "",\n  "ReviewFlag": ""\n}'
---

df_predictions rows: 100
df_validate_clean rows: 100
df_merged rows: 100


In [ ]:
'''
Notes: 

The baseline run (original prompt, no few-shot) scored Qwen at 72 vs GPT-4.1 at 82. Error analysis showed the gap is almost entirely over-# conservatism: 19 of 25 missed cases were true-Y papers wrongly marked N, concentrated in Biochemicals, Disease Regulation, and 
Fibre/Hide/Wood. Qwen treats "ecosystem service" too literally, missing that bio-inspired drugs, medical materials, and biomimetic 
synthesis count functionally. 

Fixes: 
1. add a clarification that ES contribution is functional, not locational 
2. add three real missed cases as few-shot examples
3. restore the missing "Support" option in the output format
'''

## 2. Few-shot Prompt Engineering

In [29]:
# few-shot version 1

SYSTEM_PROMPT_OPTIMIZED = """ You are an Ecosystem Service expert and a dedicated assistant designed to classify research articles (titles, keywords, abstracts given) based on the following instructions:

1. **Ecosystem Service Technology Analysis:**
Determine if the abstract describes a technological intervention that contributes to one or more of the following ecosystem services:
**Provisioning—Products obtained from ecosystems (Existing commercial market):**
- Biodiversity—The number of different species
- Food—Ingredients derived from wild and domesticated habitats
- Potable Water—Fresh water that is safe to consume
- Fuel—Materials used to generate energy
- Fibre/Hide/Wood—Materials used for clothing or construction
- Biochemicals—Molecules used in medicine
**Cultural—Benefits to quality of life and community (Existing commercial market):**
- Spiritual—Supporting the spiritual lives of people
- Recreation—Supporting the physical and mental health of people
- Aesthetic—The mental and physical health benefits of natural beauty
- Inspiration/Education—Art, music, literature, architecture, and engineering design
- Cultural Heritage—Value placed upon landscapes
- Cultural Identity—Societal identity regulated by the ecosystem (e.g., nomadic herding)
**Regulating—Benefits obtained by regulating ecosystem processes (Most amenable to technological replacement):**
- Atmospheric Regulation—Production and consumption of essential molecules (e.g., oxygen)
- Climate Regulation—Stabilization of climatic conditions
- Coastline Regulation—Stabilization of coastal lands (e.g., mangroves and reefs)
- Disease Regulation—Natural systems that reduce human disease or disease vectors
- Water Regulation—Timing and volume of water distribution across the landscape
- Waste Treatment—Filtering and treatment of waste products (incl. organics and water)
- Pollination—Distribution of pollen for the purpose of plant reproduction
**Supporting—Services that are not necessary for all other ecosystem services (Least amenable to technological replacement):**
- Soil Formation—The creation of new soil
- Nutrient Cycling—The movement of nutrients through the ecosystems
- Primary Production—The creation of sugars from sunlight

For this part:
**Decision:** Output “Y” if the abstract explicitly describes a practical technological method that contributes to one or more of these services; otherwise, output “N”.
**Category:**
If Decision is “Y”, choose **one** of the following:
- **”Support”** assists or maintains an existing natural process without intensifying it. Example: “Adding baffles so river flow still scours sediment but a little more efficiently.”
- **”Enhance”** significantly boosts the efficiency or scale of a natural process while still relying on that process. Example: “Embedding enzymes in a filter to double the nitrification rate; process still needs microbes.”
- **”Replace”** creates an artificial substitute that operates independently of the natural process. Example: “A photocatalytic panel that fixes nitrogen from air in total isolation from biological pathways.”
(If uncertain between Enhance and Replace, choose Enhance.)
Leave blank if Decision is “N”
**EcosystemService:** If Decision is “Y”, provide the exact ecosystem service from the list.
**Technology:** If Decision is “Y”, provide a concise short name for the technology used.

**How to interpret "contribution to an ecosystem service":**
A paper is Decision = "Y" whenever it presents a bio-inspired or biomimetic technology, material, molecule, or design that serves the FUNCTION of an ecosystem service — even when the work is laboratory synthesis, a medical 
material, or an engineering design rather than something deployed in nature. The link is functional, not locational. In particular, bio-inspired drugs, medical materials, and biomimetic synthesis all count as contributions to Biochemicals (molecules used in medicine).

Study these examples of papers that should be classified as "Y":

Example 1 (Y):
Title: "A milk extracellular vesicle-based nanoplatform against multidrug-resistant bacterial infections"
Output: {"Decision": "Y", "Category": "Enhance", "EcosystemService": "Biochemicals", "Technology": "Bioinspired nanoplatform", "ReviewFlag": ""}

Example 2 (Y):
Title: "Bioinspired total synthesis of natural products"
Output: {"Decision": "Y", "Category": "Replace", "EcosystemService": "Biochemicals", "Technology": "Bioinspired synthesis", "ReviewFlag": ""}

Example 3 (Y):
Title: "Bioinspired wound dressing: chitosan/polyvinyl alcohol nanofibers"
Output: {"Decision": "Y", "Category": "Enhance", "EcosystemService": "Biochemicals", "Technology": "Bioinspired nanofiber dressing", "ReviewFlag": ""}

2. **Review Paper Detection:**
Determine if the abstract indicates that the article is a review paper. If the abstract contains phrases like “review”, “survey”, “meta-analysis”, or other similar indicators, then:
- **ReviewFlag:** Set to “review”.
Otherwise, leave this field blank.

**Output Format (Strict JSON Only)**
Your output must be in JSON format only, following this structure:
{
  "Decision": "Y" or "N",
  "Category": "Support" or "Enhance" or "Replace" (leave blank if Decision = "N"),
  "EcosystemService": "(exact ecosystem service from the list)" (leave blank if Decision = "N"),
  "Technology": "(concise short name of the technology)" (leave blank if Decision = "N"),
  "ReviewFlag": "review" or ""
}
"""

In [32]:

results_opt = []

for index, row in tqdm(df_test.iterrows(), total=len(df_test),
                       desc=f"Benchmarking {TARGET_MODEL} (optimized)"):
    paper_id      = str(row['No. (number used only for testing stage)'])
    title_text    = str(row['Article Title'])
    keywords_text = str(row['Author Keywords'])
    abstract_text = str(row['Abstract'])
    combined_content = (f"Title: {title_text}\n"
                        f"Keywords: {keywords_text}\n"
                        f"Abstract: {abstract_text}")

    raw_output, status = call_with_retry(
        client, TARGET_MODEL, SYSTEM_PROMPT_OPTIMIZED, combined_content
    )
    results_opt.append({"wos_id": paper_id, "qwen_output": raw_output, "status": status})
    time.sleep(0.6)

df_predictions_opt = pd.DataFrame(results_opt)

# sanity check
print(df_predictions_opt['status'].value_counts())

Benchmarking qwen/qwen-2.5-72b-instruct (optimized): 100%|█| 100/100 [09:46<00:0

status
Success    100
Name: count, dtype: int64


In [ ]:
# df_merged.to_excel('predictions_merged.xlsx', index=False)

In [34]:

# 1. Parse JSON from the optimized run
_parsed_opt = df_predictions_opt['qwen_output'].apply(parse_full_qwen_json)
df_predictions_opt['qwen_decision_opt'] = _parsed_opt['qwen_decision']
df_predictions_opt['qwen_category_opt'] = _parsed_opt['qwen_category']
df_predictions_opt['qwen_service_opt']  = _parsed_opt['qwen_service']
df_predictions_opt['qwen_review_opt']   = _parsed_opt['qwen_review']

# 2. Clean IDs and merge (reuse the already-cleaned df_validate_clean)
df_predictions_opt['wos_id'] = (
    df_predictions_opt['wos_id'].astype(str)
    .str.replace(r'\.0$', '', regex=True).str.strip()
)

df_merged_opt = pd.merge(
    df_validate_clean,
    df_predictions_opt,
    left_on=col_id_val,
    right_on='wos_id',
    how='inner'
)

# 3. Score Qwen (optimized) using the same scoring function
df_merged_opt['qwen_score_opt'] = df_merged_opt.apply(
    lambda r: score_row(r, 'qwen_decision_opt', 'qwen_category_opt',
                        'qwen_service_opt', 'qwen_review_opt'),
    axis=1
)

# GPT score on the same rows 
df_merged_opt['gpt_score'] = df_merged_opt.apply(
    lambda r: score_row(r, 'Decision (gpt)', 'Category (gpt)',
                        'EcosystemService (gpt)', 'ReviewFlag (gpt)'),
    axis=1
)

# 4. Exclude the 3 few-shot examples (leak-free)
FEWSHOT_IDS = ['1', '33', '96']
df_eval_opt = df_merged_opt[~df_merged_opt['wos_id'].isin(FEWSHOT_IDS)].copy()

gpt_final_opt    = df_eval_opt['gpt_score'].mean() * 100
qwen_final_opt   = df_eval_opt['qwen_score_opt'].mean() * 100
author_final_opt = df_eval_opt['accuracy score '].dropna().mean() * 100

# 5. Summary
print("="*60)
print("  Optimized Prompt — Model Benchmarking (rule-based scoring)")
print(f"  Excluded {len(df_merged_opt) - len(df_eval_opt)} few-shot examples; "
      f"evaluating on {len(df_eval_opt)} papers.")
print("="*60)
print(f"  Author's manual score for GPT-4.1 : {author_final_opt:5.2f}")
print(f"  My rule-based score for GPT-4.1   : {gpt_final_opt:5.2f}")
print(f"  My rule-based score for Qwen (opt): {qwen_final_opt:5.2f}")
print(f"  GPT-4.1 vs Qwen (opt) gap         : {gpt_final_opt - qwen_final_opt:5.2f}")
print("="*60)
print("\n  Qwen (optimized) score distribution:")
print(df_eval_opt['qwen_score_opt'].value_counts().sort_index())

  Optimized Prompt — Model Benchmarking (rule-based scoring)
  Excluded 3 few-shot examples; evaluating on 97 papers.
  Author's manual score for GPT-4.1 : 85.05
  My rule-based score for GPT-4.1   : 82.47
  My rule-based score for Qwen (opt): 55.15
  GPT-4.1 vs Qwen (opt) gap         : 27.32

  Qwen (optimized) score distribution:
qwen_score_opt
0.0    41
0.5     5
1.0    51
Name: count, dtype: int64


In [35]:
# Error breakdown for the optimized run
def diagnose_opt(row):
    if row['qwen_score_opt'] == 1.0:
        return 'correct'
    h_dec = normalize_simple(row['Decision (human)'])
    q_dec = normalize_simple(row['qwen_decision_opt'])
    h_srv = normalize_service(row['EcosystemService (human)'])
    q_srv = normalize_service(row['qwen_service_opt'])
    if h_dec == 'y' and q_dec == 'n':
        return 'miss_Y_as_N (over-conservative)'
    if h_dec == 'n' and q_dec == 'y':
        return 'false_Y'
    if h_dec == 'y' and q_dec == 'y':
        return 'wrong_service' if h_srv != q_srv else 'wrong_category'
    return 'other'

df_eval_opt['error_type_opt'] = df_eval_opt.apply(diagnose_opt, axis=1)
print("\n  Qwen (optimized) error breakdown:")
print(df_eval_opt['error_type_opt'].value_counts())


  Qwen (optimized) error breakdown:
error_type_opt
correct                            51
false_Y                            25
wrong_service                      14
wrong_category                      5
miss_Y_as_N (over-conservative)     2
Name: count, dtype: int64


In [40]:
fp = df_eval_opt[df_eval_opt['error_type_opt'] == 'false_Y']
fp_ids = fp['wos_id'].tolist()
print("False Positive IDs:", fp_ids)

# Look at these papers misclassified as Y; why did humans classify them as N?
df_test['wos_id_clean'] = df_test['No. (number used only for testing stage)'].astype(str).str.replace(r'\.0$','',regex=True).str.strip()
for _, row in df_test[df_test['wos_id_clean'].isin(fp_ids)].head(5).iterrows():
    print(f"\n===== ID {row['wos_id_clean']} =====")
    print(f"Title: {row['Article Title']}")
    print(f"Abstract: {str(row['Abstract'])[:350]}...")

    q = df_eval_opt[df_eval_opt['wos_id']==row['wos_id_clean']]
    print(f"Qwen: {q['qwen_service_opt'].values}")

False Positive IDs: ['6', '8', '9', '13', '16', '19', '22', '24', '29', '36', '38', '39', '41', '46', '48', '53', '55', '57', '62', '64', '79', '80', '82', '85', '94']

===== ID 6 =====
Title: Bioinspired Synthesis of (-)-Hunterine A: Deciphering the Key Step in the Biogenetic Pathway
Abstract: A concise, bioinspired, and enantioselective synthesis of (-)-hunterine A, an odd 6/7/6/6/5 pentacyclic natural product, is described. The key step in the synthesis of this complex structure is an interim-template directed 6-exo selective epoxide ring-opening reaction, which is interwoven with a hydrolysis step of the indolenine hemiaminal template...
Qwen: <ArrowStringArray>
['Biochemicals']
Length: 1, dtype: str

===== ID 8 =====
Title: Mole-inspired Forepaw Design and Optimization Based on Resistive Force Theory
Abstract: Moles exhibit highly effective capabilities due to their unique body structures and digging techniques, making them ideal models for biomimetic research. However, a major ch

In [37]:
# Look at these typical false positives in human labeling (confirming they are all human-judged as N)
for pid in ['6', '8', '9', '16']:
    r = df_eval_opt[df_eval_opt['wos_id'] == pid]
    print(f"ID {pid}: human_decision={r['Decision (human)'].values}, "
          f"human_service={r['EcosystemService (human)'].values}, "
          f"note={r['note'].values}")

ID 6: human_decision=<ArrowStringArray>
['N']
Length: 1, dtype: str, human_service=<ArrowStringArray>
[nan]
Length: 1, dtype: str, note=<ArrowStringArray>
[nan]
Length: 1, dtype: str
ID 8: human_decision=<ArrowStringArray>
['N']
Length: 1, dtype: str, human_service=<ArrowStringArray>
[nan]
Length: 1, dtype: str, note=<ArrowStringArray>
['can be Inspiration/Education']
Length: 1, dtype: str
ID 9: human_decision=<ArrowStringArray>
['N']
Length: 1, dtype: str, human_service=<ArrowStringArray>
[nan]
Length: 1, dtype: str, note=<ArrowStringArray>
[nan]
Length: 1, dtype: str
ID 16: human_decision=<ArrowStringArray>
['N']
Length: 1, dtype: str, human_service=<ArrowStringArray>
[nan]
Length: 1, dtype: str, note=<ArrowStringArray>
[nan]
Length: 1, dtype: str


In [39]:
# Accuracy at the Decision (Y/N) level only

def decision_only(row, dec_col):
    h = normalize_simple(row['Decision (human)'])
    p = normalize_simple(row[dec_col])
    return 1.0 if h == p else 0.0

dec_acc_gpt  = df_merged.apply(lambda r: decision_only(r, 'Decision (gpt)'), axis=1).mean()*100
dec_acc_qwen = df_merged.apply(lambda r: decision_only(r, 'qwen_decision'), axis=1).mean()*100
print(f"Decision-only accuracy —  GPT: {dec_acc_gpt:.1f}   Qwen(baseline): {dec_acc_qwen:.1f}")

Decision-only accuracy —  GPT: 91.0   Qwen(baseline): 71.0


In [41]:
# few-shot version 2

SYSTEM_PROMPT_OPTIMIZED_2 = """ You are an Ecosystem Service expert and a dedicated assistant designed to classify research articles (titles, keywords, abstracts given) based on the following instructions:

1. **Ecosystem Service Technology Analysis:**
Determine if the abstract describes a technological intervention that contributes to one or more of the following ecosystem services:
**Provisioning—Products obtained from ecosystems (Existing commercial market):**
- Biodiversity—The number of different species
- Food—Ingredients derived from wild and domesticated habitats
- Potable Water—Fresh water that is safe to consume
- Fuel—Materials used to generate energy
- Fibre/Hide/Wood—Materials used for clothing or construction
- Biochemicals—Molecules used in medicine
**Cultural—Benefits to quality of life and community (Existing commercial market):**
- Spiritual—Supporting the spiritual lives of people
- Recreation—Supporting the physical and mental health of people
- Aesthetic—The mental and physical health benefits of natural beauty
- Inspiration/Education—Art, music, literature, architecture, and engineering design
- Cultural Heritage—Value placed upon landscapes
- Cultural Identity—Societal identity regulated by the ecosystem (e.g., nomadic herding)
**Regulating—Benefits obtained by regulating ecosystem processes (Most amenable to technological replacement):**
- Atmospheric Regulation—Production and consumption of essential molecules (e.g., oxygen)
- Climate Regulation—Stabilization of climatic conditions
- Coastline Regulation—Stabilization of coastal lands (e.g., mangroves and reefs)
- Disease Regulation—Natural systems that reduce human disease or disease vectors
- Water Regulation—Timing and volume of water distribution across the landscape
- Waste Treatment—Filtering and treatment of waste products (incl. organics and water)
- Pollination—Distribution of pollen for the purpose of plant reproduction
**Supporting—Services that are not necessary for all other ecosystem services (Least amenable to technological replacement):**
- Soil Formation—The creation of new soil
- Nutrient Cycling—The movement of nutrients through the ecosystems
- Primary Production—The creation of sugars from sunlight

For this part:
**Decision:** Output “Y” if the abstract explicitly describes a practical technological method that contributes to one or more of these services; otherwise, output “N”.
**Category:**
If Decision is “Y”, choose **one** of the following:
- **”Support”** assists or maintains an existing natural process without intensifying it. Example: “Adding baffles so river flow still scours sediment but a little more efficiently.”
- **”Enhance”** significantly boosts the efficiency or scale of a natural process while still relying on that process. Example: “Embedding enzymes in a filter to double the nitrification rate; process still needs microbes.”
- **”Replace”** creates an artificial substitute that operates independently of the natural process. Example: “A photocatalytic panel that fixes nitrogen from air in total isolation from biological pathways.”
(If uncertain between Enhance and Replace, choose Enhance.)
Leave blank if Decision is “N”
**EcosystemService:** If Decision is “Y”, provide the exact ecosystem service from the list.
**Technology:** If Decision is “Y”, provide a concise short name for the technology used.

**How to decide "Y" vs "N":**
A paper is "Y" only if it presents a concrete technology, material, or design that serves the FUNCTION of an ecosystem service. Being "inspired by nature" is not enough on its own. If the work is only synthesizing a molecule, building a robot, or making a device — without that output serving an ecosystem-service 
function — the Decision is "N", even if the method is bio-inspired.

Study these contrasting examples:

Example 1 (Y):
Title: "A milk extracellular vesicle-based nanoplatform against multidrug-resistant bacterial infections"
Reasoning: A bio-inspired platform delivering medicine serves the Biochemicals service (molecules used in medicine).
Output: {"Decision": "Y", "Category": "Enhance", "EcosystemService": "Biochemicals", "Technology": "Bioinspired nanoplatform", "ReviewFlag": ""}

Example 2 (Y):
Title: "Biomimetic membrane for efficient heavy-metal removal from wastewater"
Reasoning: A bio-inspired membrane that treats wastewater directly serves the Waste Treatment service.
Output: {"Decision": "Y", "Category": "Replace", "EcosystemService": "Waste Treatment", "Technology": "Biomimetic filtration membrane", "ReviewFlag": ""}

Example 3 (N):
Title: "Bioinspired synthesis of a pentacyclic natural product via epoxide ring-opening"
Reasoning: This is a bio-inspired organic synthesis METHOD. It does not, by itself, serve an ecosystem-service function, so Decision is "N".
Output: {"Decision": "N", "Category": "", "EcosystemService": "", "Technology": "", "ReviewFlag": ""}

Example 4 (N):
Title: "Mole-inspired forepaw design for digging robots based on resistive force theory"
Reasoning: A bio-inspired robot mechanism. Being inspired by an animal does not make it a contribution to any ecosystem service, so Decision is "N".
Output: {"Decision": "N", "Category": "", "EcosystemService": "", "Technology": "", "ReviewFlag": ""}

2. **Review Paper Detection:**
Determine if the abstract indicates that the article is a review paper. If the abstract contains phrases like “review”, “survey”, “meta-analysis”, or other similar indicators, then:
- **ReviewFlag:** Set to “review”.
Otherwise, leave this field blank.

**Output Format (Strict JSON Only)**
Your output must be in JSON format only, following this structure:
{
  "Decision": "Y" or "N",
  "Category": "Support" or "Enhance" or "Replace" (leave blank if Decision = "N"),
  "EcosystemService": "(exact ecosystem service from the list)" (leave blank if Decision = "N"),
  "Technology": "(concise short name of the technology)" (leave blank if Decision = "N"),
  "ReviewFlag": "review" or ""
}
"""

In [42]:

results_opt_2 = []

for index, row in tqdm(df_test.iterrows(), total=len(df_test),
                       desc=f"Benchmarking {TARGET_MODEL} (optimized)"):
    paper_id      = str(row['No. (number used only for testing stage)'])
    title_text    = str(row['Article Title'])
    keywords_text = str(row['Author Keywords'])
    abstract_text = str(row['Abstract'])
    combined_content = (f"Title: {title_text}\n"
                        f"Keywords: {keywords_text}\n"
                        f"Abstract: {abstract_text}")

    raw_output, status = call_with_retry(
        client, TARGET_MODEL, SYSTEM_PROMPT_OPTIMIZED_2, combined_content
    )
    results_opt_2.append({"wos_id": paper_id, "qwen_output": raw_output, "status": status})
    time.sleep(0.6)

df_predictions_opt_2 = pd.DataFrame(results_opt_2)

# sanity check
print(df_predictions_opt_2['status'].value_counts())

Benchmarking qwen/qwen-2.5-72b-instruct (optimized): 100%|█| 100/100 [08:26<00:0

status
Success    100
Name: count, dtype: int64


In [43]:

# 1. Parse JSON from the optimized run
_parsed_opt_2 = df_predictions_opt_2['qwen_output'].apply(parse_full_qwen_json)
df_predictions_opt_2['qwen_decision_opt_2'] = _parsed_opt_2['qwen_decision']
df_predictions_opt_2['qwen_category_opt_2'] = _parsed_opt_2['qwen_category']
df_predictions_opt_2['qwen_service_opt_2']  = _parsed_opt_2['qwen_service']
df_predictions_opt_2['qwen_review_opt_2']   = _parsed_opt_2['qwen_review']

# 2. Clean IDs and merge (reuse the already-cleaned df_validate_clean)
df_predictions_opt_2['wos_id'] = (
    df_predictions_opt_2['wos_id'].astype(str)
    .str.replace(r'\.0$', '', regex=True).str.strip()
)

df_merged_opt_2 = pd.merge(
    df_validate_clean,
    df_predictions_opt_2,
    left_on=col_id_val,
    right_on='wos_id',
    how='inner'
)

# 3. Score Qwen (optimized) using the same scoring function
df_merged_opt_2['qwen_score_opt_2'] = df_merged_opt_2.apply(
    lambda r: score_row(r, 'qwen_decision_opt_2', 'qwen_category_opt_2',
                        'qwen_service_opt_2', 'qwen_review_opt_2'),
    axis=1
)

# GPT score on the same rows 
df_merged_opt_2['gpt_score'] = df_merged_opt_2.apply(
    lambda r: score_row(r, 'Decision (gpt)', 'Category (gpt)',
                        'EcosystemService (gpt)', 'ReviewFlag (gpt)'),
    axis=1
)

# 4. Exclude the 3 few-shot examples (leak-free)
FEWSHOT_IDS = ['1', '6', '8']
df_eval_opt_2 = df_merged_opt_2[~df_merged_opt_2['wos_id'].isin(FEWSHOT_IDS)].copy()

gpt_final_opt_2    = df_eval_opt_2['gpt_score'].mean() * 100
qwen_final_opt_2   = df_eval_opt_2['qwen_score_opt_2'].mean() * 100
author_final_opt_2 = df_eval_opt_2['accuracy score '].dropna().mean() * 100

# 5. Summary
print("="*60)
print("  Optimized Prompt V2 — Model Benchmarking (rule-based scoring)")
print(f"  Excluded {len(df_merged_opt_2) - len(df_eval_opt_2)} few-shot examples; "
      f"evaluating on {len(df_eval_opt_2)} papers.")
print("="*60)
print(f"  Author's manual score for GPT-4.1 : {author_final_opt_2:5.2f}")
print(f"  My rule-based score for GPT-4.1   : {gpt_final_opt_2:5.2f}")
print(f"  My rule-based score for Qwen (opt_2): {qwen_final_opt_2:5.2f}")
print(f"  GPT-4.1 vs Qwen (opt_2) gap         : {gpt_final_opt_2 - qwen_final_opt_2:5.2f}")
print("="*60)
print("\n  Qwen (optimized) score distribution:")
print(df_eval_opt_2['qwen_score_opt_2'].value_counts().sort_index())

  Optimized Prompt V2 — Model Benchmarking (rule-based scoring)
  Excluded 3 few-shot examples; evaluating on 97 papers.
  Author's manual score for GPT-4.1 : 84.02
  My rule-based score for GPT-4.1   : 81.44
  My rule-based score for Qwen (opt_2): 70.10
  GPT-4.1 vs Qwen (opt_2) gap         : 11.34

  Qwen (optimized) score distribution:
qwen_score_opt_2
0.0    26
0.5     6
1.0    65
Name: count, dtype: int64


In [44]:
# Error breakdown for the optimized run
def diagnose_opt_2(row):
    if row['qwen_score_opt_2'] == 1.0:
        return 'correct'
    h_dec = normalize_simple(row['Decision (human)'])
    q_dec = normalize_simple(row['qwen_decision_opt_2'])
    h_srv = normalize_service(row['EcosystemService (human)'])
    q_srv = normalize_service(row['qwen_service_opt_2'])
    if h_dec == 'y' and q_dec == 'n':
        return 'miss_Y_as_N (over-conservative)'
    if h_dec == 'n' and q_dec == 'y':
        return 'false_Y'
    if h_dec == 'y' and q_dec == 'y':
        return 'wrong_service' if h_srv != q_srv else 'wrong_category'
    return 'other'

df_eval_opt_2['error_type_opt_2'] = df_eval_opt_2.apply(diagnose_opt_2, axis=1)
print("\n  Qwen (optimized_V2) error breakdown:")
print(df_eval_opt_2['error_type_opt_2'].value_counts())


  Qwen (optimized_V2) error breakdown:
error_type_opt_2
correct                            65
miss_Y_as_N (over-conservative)    19
wrong_category                      6
wrong_service                       5
false_Y                             2
Name: count, dtype: int64


In [56]:
# turned out that the qwen-2.5-70B was not good enough for such complex inference
# have to switch to another model...
# here we use "deepseek/deepseek-r1-distill-llama-70b", cause it's too slow when using "qwen/qwen3-235b-a22b-thinking-2507"
# try deepseek with prompt v2 insteal

TARGET_MODEL_1 = "deepseek/deepseek-r1-distill-llama-70b" 

def call_deepseek_with_retry(client, model, system_prompt, content, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": content}
                ],
                temperature=0.6,
                max_tokens=4096,  
            )
            if (response is None or response.choices is None
                    or len(response.choices) == 0
                    or response.choices[0].message.content is None):
                raise ValueError("Empty or malformed response")
            return response.choices[0].message.content, "Success"
        except Exception as e:
            wait = 2 ** attempt
            if attempt < max_retries - 1:
                time.sleep(wait)
            else:
                return str(e), "Failed"

# Run Inference on the Dataset

results_deepseek = []

for index, row in tqdm(df_test.iterrows(), total=len(df_test),
                       desc=f"Benchmarking {TARGET_MODEL_1}"):
    paper_id      = str(row['No. (number used only for testing stage)'])
    title_text    = str(row['Article Title'])
    keywords_text = str(row['Author Keywords'])
    abstract_text = str(row['Abstract'])

    combined_content = (f"Title: {title_text}\n"
                        f"Keywords: {keywords_text}\n"
                        f"Abstract: {abstract_text}")

    raw_output, status = call_deepseek_with_retry(
        client, TARGET_MODEL, SYSTEM_PROMPT_OPTIMIZED, combined_content
    )
    results_deepseek.append({"wos_id": paper_id, "deepseek_output": raw_output, "status": status})

    time.sleep(0.6)  


df_predictions_deepseek = pd.DataFrame(results_deepseek)

# sanity check
print(df_predictions_deepseek['status'].value_counts())

Benchmarking deepseek/deepseek-r1-distill-llama-70b: 100%|█| 100/100 [11:29<00:0

status
Success    99
Failed      1
Name: count, dtype: int64


In [57]:
# Handling 'think' tags
def parse_full_deepseek_json_safe(output_str):
    try:
        raw_text = str(output_str)
        if '</think>' in raw_text:
            raw_text = raw_text.split('</think>')[-1].strip()
            
        match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if match:
            data = json.loads(match.group())
        else:
            data = json.loads(raw_text)
            
        return pd.Series({
            'deepseek_decision': str(data.get("Decision", "")).strip(),
            'deepseek_category': str(data.get("Category", "")).strip(),
            'deepseek_service': str(data.get("EcosystemService", "")).strip(),
            'deepseek_review': str(data.get("ReviewFlag", "")).strip().lower() 
        })
    except Exception as e:
        print(f"JSON Parse Error: {e}")
        return pd.Series({'deepseek_decision': None, 'deepseek_category': None, 'deepseek_service': None, 'deepseek_review': None})

In [58]:
# Data Merging & Scoring
# 1. Parse JSON from the optimized run
_parsed_deepseek = df_predictions_deepseek['deepseek_output'].apply(parse_full_deepseek_json_safe)
df_predictions_deepseek['deepseek_decision'] = _parsed_deepseek['deepseek_decision']
df_predictions_deepseek['deepseek_category'] = _parsed_deepseek['deepseek_category']
df_predictions_deepseek['deepseek_service']  = _parsed_deepseek['deepseek_service']
df_predictions_deepseek['deepseek_review']   = _parsed_deepseek['deepseek_review']


# 2. Clean IDs and merge (reuse the already-cleaned df_validate_clean)
df_predictions_deepseek['wos_id'] = (
    df_predictions_deepseek['wos_id'].astype(str)
    .str.replace(r'\.0$', '', regex=True).str.strip()
)

df_merged_deepseek = pd.merge(
    df_validate_clean,
    df_predictions_deepseek,
    left_on=col_id_val,
    right_on='wos_id',
    how='inner'
)

# 3. Apply rule-based scoring for DeepSeek
df_merged_deepseek['deepseek_score'] = df_merged_deepseek.apply(
    lambda r: score_row(r, 'deepseek_decision', 'deepseek_category',
                        'deepseek_service', 'deepseek_review'),
    axis=1
)

# Apply rule-based scoring for GPT-4.1 (for baseline comparison)
df_merged_deepseek['gpt_score'] = df_merged_deepseek.apply(
    lambda r: score_row(r, 'Decision (gpt)', 'Category (gpt)',
                        'EcosystemService (gpt)', 'ReviewFlag (gpt)'),
    axis=1
)

# 4. evaluate on all 100 papers
gpt_final_score      = df_merged_deepseek['gpt_score'].mean() * 100
deepseek_final_score = df_merged_deepseek['deepseek_score'].mean() * 100
author_final_score   = df_merged_deepseek['accuracy score '].dropna().mean() * 100


# 5. Summary
print("="*60)
print(f"  Model Benchmarking (rule-based scoring) — {TARGET_MODEL_1}")
print("="*60)
print(f"  Author's manual score for GPT-4.1 : {author_final_score:5.2f}")
print(f"  My rule-based score for GPT-4.1   : {gpt_final_score:5.2f}")
print(f"  My rule-based score for DeepSeek-R1 : {deepseek_final_score:5.2f}")
print(f"  GPT-4.1 vs DeepSeek-R1 (v1) gap          : {gpt_final_score - deepseek_final_score:5.2f}")
print("="*60)
print("\n DeepSeek-R1 score distribution:")
print(df_merged_deepseek['deepseek_score'].value_counts().sort_index())

JSON Parse Error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
  Model Benchmarking (rule-based scoring) — deepseek/deepseek-r1-distill-llama-70b
  Author's manual score for GPT-4.1 : 84.50
  My rule-based score for GPT-4.1   : 82.00
  My rule-based score for DeepSeek-R1 : 55.00
  GPT-4.1 vs DeepSeek-R1 (v1) gap          : 27.00

 DeepSeek-R1 score distribution:
deepseek_score
0.0    42
0.5     6
1.0    52
Name: count, dtype: int64
